# 04. 덤핑방지 관세 규제 이벤트 전처리

GitHub Issue #1의 설계를 노트북으로 구현한다.

목표:
- `data/raw`의 산업통상부 덤핑방지 관세 부과 현황 CSV를 로드한다.
- 복수 국가명을 국가 단위 long format으로 분리한다.
- 품목명과 관세부과범위를 정규화한다.
- `regulation_events`, `product_hs_mapping`, `regulation_event_candidates` 산출물을 생성한다.
- 현재 데이터로 확인 가능한 기초 집계를 만든다.

주의:
- 파일 탐색과 원천 CSV 로드는 `src.data.loaders`의 공통 유틸을 사용한다.
- 국가명 정규화, 관세범위 파싱, 후보 생성은 분석 설계가 바뀔 수 있으므로 노트북에 둔다.
- raw 데이터에는 HS 코드가 없으므로 `product_hs_mapping`은 수동 보강용 템플릿으로 생성한다.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

from src.data.loaders import load_antidumping_from_raw_dir, normalize_unicode_text

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

## 1. 경로 및 원천 데이터 로드

macOS 한글 파일명은 NFC/NFD 차이로 문자열 glob이 실패할 수 있으므로, `src.data.loaders.load_antidumping_from_raw_dir()`로 `data/raw`의 CSV 탐색과 로드를 함께 처리한다.

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

raw_csv_candidates = sorted(RAW_DIR.glob("*.csv"))
raw_csv_candidates

[PosixPath('/Users/leejungyeon/Workspace/projects/무역통계경진대회/trade-circumvention-monitor/data/raw/산업통상부_무역구제 덤핑방지 관세 부과 현황_20251231.csv')]

In [3]:
RAW_CSV_PATH, raw_df = load_antidumping_from_raw_dir(RAW_DIR)

print(RAW_CSV_PATH)
print(raw_df.shape)
raw_df.head()

/Users/leejungyeon/Workspace/projects/무역통계경진대회/trade-circumvention-monitor/data/raw/산업통상부_무역구제 덤핑방지 관세 부과 현황_20251231.csv
(160, 7)


,source_row_id,국가명,품목,관세부과범위,부과시작일,부과종료일,관련법령
0,1,"일본,대만",D.C.P,가격약속,1989-01-13,1989-12-31,재무부 공고 제89-2호
1,2,프랑스,알루미나 시멘트,가격약속,1989-09-01,1991-08-30,재무부 고시 제89-14호
2,3,"미국,일본",폴리아세탈수지,기준가격과 수입가격의 차액,1991-09-30,1993-08-30,대통령령 제13467호
3,4,중국,정제인산,40.46~54.28,1993-02-20,1996-02-19,대통령령 제13846호
4,5,태국,볼베어링,6.27,1993-01-28,1998-01-27,대통령령 제13873호


## 2. 칼럼 이해와 국가명 분리

원천 데이터의 주요 컬럼은 다음 의미로 해석한다.

1. `국가명`: 덤핑방지관세가 적용되는 수입 물품의 원산지 또는 수출국을 의미한다. 예를 들어 첫 번째 행의 `일본,대만`은 일본산 또는 대만산으로 수입될 때 덤핑방지관세 대상이라는 뜻이다. 따라서 이 노트북에서는 목적지나 추천 대상이라는 의미의 `target_country_*`가 아니라 `origin_country_*` 컬럼명으로 정규화한다.
2. `품목`: 덤핑방지관세가 부과되는 물품명이다.
3. `관세부과범위`: 해당 품목에 적용되는 덤핑방지관세율 또는 부과 방식이다.
   - 숫자 하나만 있는 경우: 해당 비율의 덤핑방지관세율이 적용된다.
   - 범위로 표시된 경우: 같은 국가/품목이라도 기업별, 수출자별, 생산자별로 적용 관세율이 다를 수 있다.
   - `가격약속`: 관세 부과 대신 또는 관세와 함께, 수출자가 일정 가격 이하로 수출하지 않겠다고 약속한 조치가 포함되었다는 의미다.
   - `일부부과제외`: 일부 세부 품목, 업체, 규격은 덤핑방지관세 부과 대상에서 제외되었다는 의미다.
   - `수출중지약속`: 해당 외국 수출자가 문제 품목을 한국으로 더 이상 수출하지 않겠다고 약속한 것이다.
4. `부과시작일`: 덤핑방지관세 부과 시작일이다.
5. `부과종료일`: 덤핑방지관세 부과 종료일이다.
6. `관련법령`: 관세 조치의 법적 근거 문서 번호다. 정확한 부과 대상, 예외 품목, 업체별 세율, 가격약속 조건은 이 번호를 기준으로 원문을 확인해야 한다.

아래에서는 `국가명`에 여러 국가가 쉼표로 묶인 경우 원산지 국가별 이벤트 행으로 확장한다. 일부 약칭과 표기 차이는 분석용 표준명으로 통일한다.


In [4]:
COUNTRY_SYNONYMS = {
    "인니": "인도네시아",
    "말레이지아": "말레이시아",
    "아랍에미레이트": "아랍에미리트",
}


def split_country_cell(value: str) -> list[str]:
    text = normalize_unicode_text(value).strip().replace("，", ",")
    if text == "미국캐나다":
        return ["미국", "캐나다"]
    return [part.strip() for part in text.split(",") if part.strip()]


def standardize_country_name(value: str) -> str:
    text = normalize_unicode_text(value).strip()
    return COUNTRY_SYNONYMS.get(text, text)


def normalize_country_names(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        origin_country_group_raw = normalize_unicode_text(row["국가명"])
        countries = split_country_cell(origin_country_group_raw)
        for seq, country in enumerate(countries, start=1):
            rows.append(
                {
                    **row.to_dict(),
                    "origin_country_group_raw": origin_country_group_raw,
                    "origin_country_seq": seq,
                    "origin_country_name_raw": country,
                    "origin_country_name_kr": standardize_country_name(country),
                }
            )
    return pd.DataFrame(rows)


country_long_df = normalize_country_names(raw_df)
country_long_df[
    [
        "source_row_id",
        "origin_country_group_raw",
        "origin_country_seq",
        "origin_country_name_raw",
        "origin_country_name_kr",
    ]
].head(12)


,source_row_id,origin_country_group_raw,origin_country_seq,origin_country_name_raw,origin_country_name_kr
0,1,"일본,대만",1,일본,일본
1,1,"일본,대만",2,대만,대만
2,2,프랑스,1,프랑스,프랑스
3,3,"미국,일본",1,미국,미국
4,3,"미국,일본",2,일본,일본
5,4,중국,1,중국,중국
6,5,태국,1,태국,태국
7,6,중국,1,중국,중국
8,7,일본,1,일본,일본
9,8,"미국,일본,대만",1,미국,미국


## 3. 품목명 정규화 및 관세부과범위 파싱

품목명은 재심 표기와 괄호 설명을 분리하고, 관세부과범위는 숫자 관세율과 비정형 조치 유형으로 나눈다.

In [5]:
REVIEW_PATTERN = re.compile(r"\(?\s*\d+차\s*재심\)?")
PAREN_PATTERN = re.compile(r"\([^)]*\)")
NUMBER_PATTERN = re.compile(r"\d+(?:\.\d+)?")

DUTY_TYPE_KR = {
    "ad_valorem": "세율부과",
    "price_undertaking": "가격약속",
    "reference_price_diff": "기준가격차액",
    "partial_exclusion": "일부부과제외",
    "export_stop_undertaking": "수출중지약속",
    "mixed": "복합",
    "unknown": "확인필요",
}


def normalize_product_name(value: str) -> str:
    text = normalize_unicode_text(value).strip()
    text = REVIEW_PATTERN.sub("", text)
    text = PAREN_PATTERN.sub("", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_duty_scope_value(value: str) -> dict[str, object]:
    text = normalize_unicode_text(value).strip()
    numbers = [float(match) for match in NUMBER_PATTERN.findall(text)]

    has_numeric_rate = bool(numbers)
    has_price_undertaking = "가격약속" in text
    has_reference_price_diff = "기준가격" in text and "수입가격" in text and "차액" in text
    has_partial_exclusion = "일부부과제외" in text
    has_export_stop_undertaking = "수출중지약속" in text

    flags = [
        has_numeric_rate,
        has_price_undertaking,
        has_reference_price_diff,
        has_partial_exclusion,
        has_export_stop_undertaking,
    ]
    if sum(flags) > 1:
        duty_type = "mixed"
    elif has_numeric_rate:
        duty_type = "ad_valorem"
    elif has_price_undertaking:
        duty_type = "price_undertaking"
    elif has_reference_price_diff:
        duty_type = "reference_price_diff"
    elif has_partial_exclusion:
        duty_type = "partial_exclusion"
    elif has_export_stop_undertaking:
        duty_type = "export_stop_undertaking"
    else:
        duty_type = "unknown"

    return {
        "duty_text_raw": text,
        "duty_type": duty_type,
        "duty_type_kr": DUTY_TYPE_KR[duty_type],
        "duty_rate_min": min(numbers) if numbers else np.nan,
        "duty_rate_max": max(numbers) if numbers else np.nan,
    }


parsed_duty_df = country_long_df["관세부과범위"].apply(parse_duty_scope_value).apply(pd.Series)

normalized_df = pd.concat([country_long_df, parsed_duty_df], axis=1)
normalized_df["product_name_kr"] = normalized_df["품목"].map(normalize_unicode_text)
normalized_df["product_name_normalized"] = normalized_df["품목"].map(normalize_product_name)

normalized_df[["품목", "product_name_normalized", "관세부과범위", "duty_type_kr", "duty_rate_min", "duty_rate_max"]].head(15)


,품목,product_name_normalized,관세부과범위,duty_type_kr,duty_rate_min,duty_rate_max
0,D.C.P,D.C.P,가격약속,가격약속,NaN,NaN
1,D.C.P,D.C.P,가격약속,가격약속,NaN,NaN
2,알루미나 시멘트,알루미나 시멘트,가격약속,가격약속,NaN,NaN
3,폴리아세탈수지,폴리아세탈수지,기준가격과 수입가격의 차액,기준가격차액,NaN,NaN
4,폴리아세탈수지,폴리아세탈수지,기준가격과 수입가격의 차액,기준가격차액,NaN,NaN
5,정제인산,정제인산,40.46~54.28,세율부과,40.46,54.28
6,볼베어링,볼베어링,6.27,세율부과,6.27,6.27
7,소다회,소다회,"66.11,가격약속",복합,66.11,66.11
8,PS 인쇄판,PS 인쇄판,24.51~38.16,세율부과,24.51,38.16
9,유리장섬유,유리장섬유,"10.3~58.7,가격약속",복합,10.30,58.70


## 4. 정규화 테이블 생성

최종 정규화 테이블은 원자료를 너무 많이 변형하지 않고, EDA에 바로 필요한 최소 컬럼만 남긴다.

- 여러 국가가 한 칸에 있는 `국가명`은 국가별 행으로 분리한다.
- 품목명은 원문을 보존하되, 묶어서 보기 쉬운 정규화 품목명을 함께 둔다.
- `관세부과범위`는 원문을 보존하고, EDA용 관세유형과 숫자 세율 범위만 추출한다.
- `event_id`, HS 후보, 분석 우선순위처럼 다음 단계 분석 설계에 가까운 컬럼은 만들지 않는다.


In [6]:
def extract_date_from_filename(path: Path) -> pd.Timestamp:
    match = re.search(r"(20\d{6})", path.name)
    if match:
        return pd.to_datetime(match.group(1), format="%Y%m%d")
    return pd.Timestamp.today().normalize()


EXTRACT_AS_OF = extract_date_from_filename(RAW_CSV_PATH)

eda_base = normalized_df.copy()
eda_base["start_date"] = pd.to_datetime(eda_base["부과시작일"])
eda_base["end_date"] = pd.to_datetime(eda_base["부과종료일"])
eda_base["duration_days"] = (eda_base["end_date"] - eda_base["start_date"]).dt.days + 1
eda_base["is_active_as_of_extract"] = (eda_base["start_date"] <= EXTRACT_AS_OF) & (eda_base["end_date"] >= EXTRACT_AS_OF)
eda_base["legal_basis_clean"] = eda_base["관련법령"].map(normalize_unicode_text)

antidumping_normalized = pd.DataFrame(
    {
        "원천행번호": eda_base["source_row_id"],
        "국가명_원문": eda_base["origin_country_group_raw"],
        "국가분리순번": eda_base["origin_country_seq"],
        "국가명_정규화": eda_base["origin_country_name_kr"],
        "품목명_원문정리": eda_base["product_name_kr"],
        "품목명_정규화": eda_base["product_name_normalized"],
        "관세부과범위_원문": eda_base["duty_text_raw"],
        "관세유형": eda_base["duty_type_kr"],
        "관세율_최소": eda_base["duty_rate_min"],
        "관세율_최대": eda_base["duty_rate_max"],
        "부과시작일": eda_base["start_date"],
        "부과종료일": eda_base["end_date"],
        "부과기간일": eda_base["duration_days"],
        "관련법령": eda_base["legal_basis_clean"],
        "추출기준일_유효여부": eda_base["is_active_as_of_extract"],
    }
).sort_values(["부과시작일", "원천행번호", "국가분리순번"]).reset_index(drop=True)

print(f"extract_as_of: {EXTRACT_AS_OF.date()}")
print(antidumping_normalized.shape)
antidumping_normalized.head(15)


extract_as_of: 2025-12-31
(275, 15)


,원천행번호,국가명_원문,국가분리순번,국가명_정규화,품목명_원문정리,품목명_정규화,관세부과범위_원문,관세유형,관세율_최소,관세율_최대,부과시작일,부과종료일,부과기간일,관련법령,추출기준일_유효여부
0,1,"일본,대만",1,일본,D.C.P,D.C.P,가격약속,가격약속,NaN,NaN,1989-01-13,1989-12-31,353,재무부 공고 제89-2호,False
1,1,"일본,대만",2,대만,D.C.P,D.C.P,가격약속,가격약속,NaN,NaN,1989-01-13,1989-12-31,353,재무부 공고 제89-2호,False
2,2,프랑스,1,프랑스,알루미나 시멘트,알루미나 시멘트,가격약속,가격약속,NaN,NaN,1989-09-01,1991-08-30,729,재무부 고시 제89-14호,False
3,3,"미국,일본",1,미국,폴리아세탈수지,폴리아세탈수지,기준가격과 수입가격의 차액,기준가격차액,NaN,NaN,1991-09-30,1993-08-30,701,대통령령 제13467호,False
4,3,"미국,일본",2,일본,폴리아세탈수지,폴리아세탈수지,기준가격과 수입가격의 차액,기준가격차액,NaN,NaN,1991-09-30,1993-08-30,701,대통령령 제13467호,False
5,5,태국,1,태국,볼베어링,볼베어링,6.27,세율부과,6.27,6.27,1993-01-28,1998-01-27,1826,대통령령 제13873호,False
6,4,중국,1,중국,정제인산,정제인산,40.46~54.28,세율부과,40.46,54.28,1993-02-20,1996-02-19,1095,대통령령 제13846호,False
7,7,일본,1,일본,PS 인쇄판,PS 인쇄판,24.51~38.16,세율부과,24.51,38.16,1993-11-08,1998-11-07,1826,대통령령 제14046호,False
8,6,중국,1,중국,소다회,소다회,"66.11,가격약속",복합,66.11,66.11,1993-12-31,1996-12-30,1096,대통령령 제14047호,False
9,8,"미국,일본,대만",1,미국,유리장섬유,유리장섬유,"10.3~58.7,가격약속",복합,10.30,58.70,1994-04-12,1999-04-11,1826,재무부령 제1997호,False


### 정규화 컬럼 설명

| 컬럼 | 의미 |
| --- | --- |
| `원천행번호` | 원자료에서 몇 번째 행이었는지 나타내는 번호 |
| `국가명_원문` | 원자료의 `국가명` 값 그대로. 예: `일본,대만` |
| `국가분리순번` | 한 행에 여러 국가가 있을 때 분리된 순서 |
| `국가명_정규화` | 약칭과 표기 차이를 정리한 국가명. 예: `인니` -> `인도네시아` |
| `품목명_원문정리` | 원자료 품목명을 유니코드/공백만 정리한 값 |
| `품목명_정규화` | 재심 표기와 괄호 설명을 제거해 같은 품목끼리 묶기 쉽게 만든 값 |
| `관세부과범위_원문` | 원자료의 관세부과범위 문구 |
| `관세유형` | 관세부과범위를 `세율부과`, `가격약속`, `일부부과제외`, `복합`, `확인필요` 등으로 단순 분류한 값 |
| `관세율_최소`, `관세율_최대` | 관세부과범위에서 숫자로 읽히는 세율의 최소/최대값 |
| `부과시작일`, `부과종료일` | 날짜형으로 변환한 부과 기간 |
| `부과기간일` | 부과시작일~부과종료일 사이 일수 |
| `관련법령` | 조치의 법적 근거 번호 |
| `추출기준일_유효여부` | 파일명 기준일 현재 조치가 유효한지 여부 |


## 5. 저장

정규화 결과만 저장한다. EDA 표는 노트북에서 바로 확인하고, 필요할 때 별도 저장한다.


In [7]:
ANTIDUMPING_NORMALIZED_PATH = INTERIM_DIR / "antidumping_normalized.csv"

antidumping_normalized.to_csv(ANTIDUMPING_NORMALIZED_PATH, index=False, encoding="utf-8-sig")

print(ANTIDUMPING_NORMALIZED_PATH.relative_to(PROJECT_ROOT))


data/interim/antidumping_normalized.csv


## 6. EDA

원자료에서 바로 확인할 수 있는 기본 질문만 본다.

1. 원자료 행이 국가 분리 후 몇 행으로 늘어났는가?
2. 어떤 국가가 가장 자주 등장하는가?
3. 어떤 품목이 가장 자주 등장하는가?
4. 관세유형은 어떻게 나뉘는가?
5. 연도별 부과 시작 건수는 어떻게 변하는가?
6. 현재 유효한 조치는 무엇인가?


In [8]:
summary = {
    "원자료_행수": len(raw_df),
    "국가분리후_행수": len(antidumping_normalized),
    "국가수": antidumping_normalized["국가명_정규화"].nunique(),
    "품목수": antidumping_normalized["품목명_정규화"].nunique(),
    "현재유효_건수": int(antidumping_normalized["추출기준일_유효여부"].sum()),
    "평균부과기간일": round(antidumping_normalized["부과기간일"].mean(), 1),
}

pd.Series(summary, name="값").to_frame()


,값
원자료_행수,160.0
국가분리후_행수,275.0
국가수,27.0
품목수,72.0
현재유효_건수,44.0
평균부과기간일,1324.2


In [9]:
country_counts = (
    antidumping_normalized["국가명_정규화"]
    .value_counts()
    .rename_axis("국가명")
    .reset_index(name="건수")
)

country_counts.head(20)


,국가명,건수
0,중국,88
1,일본,45
2,대만,19
3,미국,18
4,인도,18
5,말레이시아,12
6,태국,11
7,인도네시아,11
8,싱가포르,8
9,베트남,7


In [10]:
product_counts = (
    antidumping_normalized["품목명_정규화"]
    .value_counts()
    .rename_axis("품목명")
    .reset_index(name="건수")
)

product_counts.head(20)


,품목명,건수
0,스테인레스스틸바,15
1,염화콜린,14
2,초산에틸,14
3,PET필름,14
4,합판,14
5,유리장섬유,12
6,에탄올아민,9
7,폴리에스테르 장섬유 연신가공사,9
8,OPP필름,9
9,페로실리코망간,8


In [11]:
duty_type_counts = (
    antidumping_normalized["관세유형"]
    .value_counts()
    .rename_axis("관세유형")
    .reset_index(name="건수")
)

duty_type_counts


,관세유형,건수
0,세율부과,210
1,복합,44
2,일부부과제외,10
3,가격약속,7
4,기준가격차액,2
5,수출중지약속,1
6,확인필요,1


In [12]:
yearly_start_counts = (
    antidumping_normalized.assign(부과시작연도=antidumping_normalized["부과시작일"].dt.year)
    .groupby("부과시작연도")
    .size()
    .reset_index(name="건수")
)

yearly_start_counts.tail(20)


,부과시작연도,건수
14,2006,4
15,2007,3
16,2008,12
17,2009,4
18,2010,11
19,2011,4
20,2012,11
21,2013,15
22,2014,9
23,2015,8


In [13]:
duty_rate_summary = antidumping_normalized[["관세율_최소", "관세율_최대"]].describe()
duty_rate_summary


,관세율_최소,관세율_최대
count,254.000000,254.000000
mean,11.427638,26.369134
std,11.287020,16.378407
min,2.150000,4.460000
25%,4.030000,15.382500
50%,7.945000,25.000000
75%,13.425000,34.902500
max,72.410000,100.100000


In [14]:
active_measures = antidumping_normalized[antidumping_normalized["추출기준일_유효여부"]].sort_values(
    ["부과종료일", "국가명_정규화", "품목명_정규화"]
)

active_measures[
    [
        "국가명_정규화",
        "품목명_정규화",
        "관세유형",
        "관세율_최소",
        "관세율_최대",
        "부과시작일",
        "부과종료일",
        "관련법령",
    ]
].head(30)


,국가명_정규화,품목명_정규화,관세유형,관세율_최소,관세율_최대,부과시작일,부과종료일,관련법령
227,중국,H형강,세율부과,28.23,32.72,2021-03-30,2026-03-29,기획재정부령 제851호
228,중국,인쇄제판용 평면모양 사진 플레이트,세율부과,8.78,10.32,2021-05-03,2026-05-02,기획재정부령 제882호
232,대만,PET필름,세율부과,3.19,60.95,2021-12-24,2026-12-23,기획재정부령 제877호
234,아랍에미리트,PET필름,세율부과,3.19,60.95,2021-12-24,2026-12-23,기획재정부령 제877호
233,태국,PET필름,세율부과,3.19,60.95,2021-12-24,2026-12-23,기획재정부령 제877호
235,중국,폴리에스테르 장섬유 완전연신사,세율부과,3.95,10.91,2022-01-06,2027-01-05,기획재정부령 제888호
236,중국,플로트판유리,세율부과,36.01,36.01,2022-04-08,2027-04-07,기획재정부령 제913호
237,미국,부틸글리콜에테르,세율부과,20.10,25.00,2022-07-26,2027-07-25,기획재정부령 제925호
238,프랑스,부틸글리콜에테르,세율부과,20.10,25.00,2022-07-26,2027-07-25,기획재정부령 제925호
240,중국,인쇄제판용 평면모양 사진 플레이트,세율부과,3.60,7.61,2022-10-25,2027-10-24,기획재정부령 제940호
